# Quantum Data Representation and Variational Circuits

This notebook explores how to map classical data into quantum states using different encoding techniques and demonstrates core quantum effects using both **PennyLane** and **Qiskit**.

### Feature Analysis
- **Breast Cancer**: 30 features
- **Ionosphere**: 34 features
- **Credit Card Fraud**: 30 features
- **NSL-KDD**: ~40 numeric features

To process these on current quantum simulators or NISQ hardware, we either use **PCA** to reduce features to the number of qubits (Angle Encoding) or use **Amplitude Encoding** to pack all features into a few qubits.

In [1]:
import numpy as np
import pennylane as qml
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Statevector
import matplotlib.pyplot as plt

# Sample data (4 features normalized)
data_point = np.array([0.1, 0.5, 0.8, 0.2])
normalized_data = data_point / np.linalg.norm(data_point) # For amplitude encoding

## 1. Encoding Techniques & Trade-offs

| Method | Qubits | Circuit Depth | Complexity | Use Case |
| :--- | :--- | :--- | :--- | :--- |
| **Basis** | $N$ | $O(1)$ | Low | Categorical/Binary Data |
| **Angle** | $N$ | $O(1)$ | Low | Continuous data, easy to train |
| **Amplitude** | $\log_2(N)$ | $O(N)$ | High | High-dimensional data, limited qubits |

In [2]:
def show_encodings_pennylane(data):
    n_qubits = len(data)
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev)
    def angle_encoding_circuit(inputs):
        qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation='X')
        return qml.state()

    @qml.qnode(qml.device("default.qubit", wires=int(np.log2(len(data)))))
    def amplitude_encoding_circuit(inputs):
        qml.AmplitudeEmbedding(inputs, wires=range(int(np.log2(len(data)))), normalize=True)
        return qml.state()

    print("PennyLane Angle Encoding State:\n", angle_encoding_circuit(data)[:4])
    print("PennyLane Amplitude Encoding State:\n", amplitude_encoding_circuit(data))

show_encodings_pennylane(data_point)

PennyLane Angle Encoding State:
 [ 0.88685929+0.j          0.        -0.08898274j  0.        -0.37495809j
 -0.0376213 +0.j        ]
PennyLane Amplitude Encoding State:
 [0.10314212+0.j 0.51571062+0.j 0.825137  +0.j 0.20628425+0.j]


## 2. Quantum Effects: Superposition, Entanglement, and Interference

We use **Qiskit** to demonstrate how data becomes "quantum".

In [3]:
def quantum_effects_demo():
    # 2 Qubits
    qc = QuantumCircuit(2)
    
    # 1. Superposition (Hadamard gate)
    qc.h(0)
    
    # 2. Entanglement (CNOT gate)
    qc.cx(0, 1)
    
    # 3. Interference (Applying another H to qubit 0 can reverse superposition)
    # This is often used in algorithms like Grover's or for phase estimation
    
    print("Quantum Circuit demonstrating Superposition & Entanglement:")
    print(qc.draw())
    
    state = Statevector.from_instruction(qc)
    print("\nFinal State Vector (Bell State):")
    print(state.data)

quantum_effects_demo()

Quantum Circuit demonstrating Superposition & Entanglement:
     ┌───┐     
q_0: ┤ H ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘

Final State Vector (Bell State):
[0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]


## 3. Variational Quantum Circuit (VQC)

A VQC consists of a fixed data-loading part (Encoder) and a trainable variational part (Ansatz).

In [4]:
n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def vqc_circuit(data, weights):
    # Encoder
    qml.AngleEmbedding(data, wires=range(n_qubits))
    
    # Ansatz (Trainable Layers)
    for W in weights:
        for i in range(n_qubits):
            qml.Rot(W[i, 0], W[i, 1], W[i, 2], wires=i)
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])
            
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

# Random weights for a 2-layer Ansatz
weights = np.random.random(size=(2, n_qubits, 3))
output = vqc_circuit(data_point, weights)
print("VQC Output (Expectation Values):", output)

VQC Output (Expectation Values): [np.float64(0.6826695112966286), np.float64(0.390367835439811), np.float64(0.1699910530883577), np.float64(0.2044052247174784)]


In [5]:
print("VQC Circuit Diagram:")
print(qml.draw(vqc_circuit)(data_point, weights))

VQC Circuit Diagram:
0: ─╭AngleEmbedding(M0)──Rot(0.16,0.66,0.63)─╭●──Rot(0.94,0.44,0.18)───────────────────── ···
1: ─├AngleEmbedding(M0)──Rot(0.43,0.07,0.81)─╰X─╭●────────────────────Rot(0.91,0.60,0.18) ···
2: ─├AngleEmbedding(M0)──Rot(0.15,0.74,0.80)────╰X───────────────────╭●────────────────── ···
3: ─╰AngleEmbedding(M0)──Rot(0.44,0.92,0.71)─────────────────────────╰X────────────────── ···

0: ··· ─╭●─────────────────────────┤  <Z>
1: ··· ─╰X───────────────────╭●────┤  <Z>
2: ··· ──Rot(0.24,0.29,0.28)─╰X─╭●─┤  <Z>
3: ··· ──Rot(0.98,0.11,0.40)────╰X─┤  <Z>

M0 = 
[0.1 0.5 0.8 0.2]
